In [98]:
import xarray as xr
import rioxarray
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score
from scipy.ndimage import median_filter, minimum_filter, maximum_filter, uniform_filter
import lightgbm as lgb 
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib
from scipy import stats


In [54]:
payload = joblib.load('trained-ashanti-2021.joblib')
X_train = payload['X_train']
y_train = payload['y_train']
metadata_df_train = payload['metadata_df_train']
X_test = payload['X_test']
y_test = payload['y_test']
metadata_df_test = payload['metadata_df_test']
clf = payload['clf']
user_attrs = payload['user_attrs']

metadata_df_test.head(1)

,lon,lat,fold_idx
131277,-1.326525,7.621186,831


# Real area from Cocoa area estimation

My issue is that I'm dealing with 2 back-to-back area estimation.
Hence, for each region, I have

1. The real cocoa area
2. The ETHZ's cocoa area estimation
3. My cocoa area estimation

First, I need to figure out how to estimate the real cocoa area from the ETHZ's cocoa area estimation.

## ETHZ's estimate

### Naive Area Estimate

$$A_{ETHZ} = Positive_{ETHZ} = TP_{ETHZ} + FP_{ETHZ}$$

$$Â^{naive}_{ETHZ} = A_{ETHZ}$$


In [87]:
# Cocoa area according to the ETHZ, naively

def compute_naive_area_proportion(y) -> float:
    """Compute the naive area (i.e. pixel counting) given a list of prediction."""
    return float(y.sum() / len(y))

def print_ethz_naive_area_proportion(region: str) -> float:
    # Load the ground truth, as it was split
    payload = joblib.load(f'trained-{region.lower()}-2021.joblib')
    y_test = payload['y_test']

    # Figure out area proportion
    area_proportion = compute_naive_area_proportion(y_test)

    print(f"{region}'s test subregion: ETHZ's cocoa area proportion (naive): {area_proportion * 100:.3f}%")

print_ethz_naive_area_proportion('ashanti')
print_ethz_naive_area_proportion('western north')

ashanti's test subregion: ETHZ's cocoa area proportion (naive): 72.332%
western north's test subregion: ETHZ's cocoa area proportion (naive): 97.941%


### Bias correction

$$A_{ETHZ} = Positive_{ETHZ} = TP_{ETHZ} + FP_{ETHZ} = \frac{TP_{ETHZ}}{Precision_{ETHZ}}$$
$$A_{true} = TP_{ETHZ} + FN_{ETHZ} = \frac{TP_{ETHZ}}{Recall_{ETHZ}}$$

$$\rightarrow A_{true} = A_{ETHZ} * \frac{Precision_{ETHZ}}{Recall_{ETHZ}}$$

$A_{true}$ is the actual cocoa area.\
The thing is, our precision/recall is estimated off some subset (i.e. our test set).\
Hence we make the assumption that our test set's Precision/Recall matches the actual Precision/Recall of our model.

$$\rightarrow Â^{corrected}_{ETHZ} = A_{ETHZ} * \frac{Precision^{test}_{ETHZ}}{Recall^{test}_{ETHZ}}$$

In [97]:
def compute_corrected_area_proportion(y, precision, recall):
    area_proportion = y.sum() / len(y)
    return min(area_proportion * precision / recall, 1.0)

def compute_ethz_corrected_area_proportion(region: str) -> float:
    """Compute the Corrected Area Proportion (CAP) off the ETHZ's prediction, taking only the region's testing area into account."""
    # Load the ground truth, as it was split
    payload = joblib.load(f'trained-{region.lower()}-2021.joblib')
    y_test = payload['y_test']

    # Figure out area proportion
    return compute_corrected_area_proportion(y_test, precision=0.917, recall=0.909) # Precision/Recall from ETHZ study

def print_ethz_corrected_area_proportion(region: str) -> float:
    """Print the Corrected Area Proportion (CAP) off the ETHZ's prediction, taking only the region's testing area into account."""
    area_proportion = compute_ethz_corrected_area_proportion(region)
    print(f"{region}'s test subregion: ETHZ's cocoa area proportion (corrected): {area_proportion * 100:.3f}%")


print_ethz_corrected_area_proportion('ashanti')
print_ethz_corrected_area_proportion('western north')
print_ethz_corrected_area_proportion('western')

ashanti's test subregion: ETHZ's cocoa area proportion (corrected): 72.969%
western north's test subregion: ETHZ's cocoa area proportion (corrected): 98.803%
western's test subregion: ETHZ's cocoa area proportion (corrected): 95.756%


## Our estimate

The ETHZ's estimated area are the best we'll get.
We make the assumption that the ETHZ's estimate is correct, i.e

$$Â^{corrected}_{ETHZ} = A_{true}$$

From them, we'll compare our estimate, i.e.

$$A_{true} = Â^{corrected}_{ETHZ} = A_{LGBM} * \frac{Precision^{Val}_{LGBM}}{Recall^{Val}_{LGBM}}$$
$$\rightarrow Â^{corrected}_{LGBM} = A_{LGBM} * \frac{Precision^{Val}_{LGBM}}{Recall^{Val}_{LGBM}}$$

We'll measure how far off $Â^{corrected}_{ETHZ}$ we are

In [106]:
def print_our_corrected_area_proportion(region: str, ci_level: float = 0.95, n_bootstraps: int = 1000) -> float:
    """Print the CAP with a Bootstrapped Confidence Interval capturing both data and seed variance."""
    # Load payload
    payload = joblib.load(f'trained-{region.lower()}-2021.joblib')
    X_test = payload['X_test']
    user_attrs = payload['user_attrs']
    clfs = payload['clf']

    # Figure out precision/recall via micro-avg (Constants from CV)
    precision = (np.array(user_attrs['precisions']) * np.array(user_attrs['val_sizes'])).sum() / sum(user_attrs["val_sizes"])
    recall = (np.array(user_attrs['recalls']) * np.array(user_attrs['val_sizes'])).sum() / sum(user_attrs["val_sizes"])

    # 1. Pre-compute predictions for all models to save time
    # Shape will be (n_models, n_test_samples)
    all_yhats = np.array([clf.predict(X_test) for clf in clfs])
    n_samples = X_test.shape[0]
    
    bootstrapped_proportions = []

    # 2. The Bootstrap Loop
    # We set a random seed here just for reproducibility of the bootstrap itself
    np.random.seed(42) 
    
    for _ in range(n_bootstraps):
        # Draw random indices with replacement
        boot_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        # For this specific virtual test set, calculate the metric across all our models
        for yhats in all_yhats:
            # Subset the predictions to match our bootstrapped sample
            yhat_boot = yhats[boot_indices]
            
            # Compute the area proportion for this specific model on this specific bootstrap sample
            ap = compute_corrected_area_proportion(yhat_boot, precision=precision, recall=recall)
            bootstrapped_proportions.append(ap)

    # 3. Calculate Percentile-based Confidence Interval
    bootstrapped_proportions = np.array(bootstrapped_proportions)
    
    mean_ap = np.mean(bootstrapped_proportions)
    
    # Calculate the lower and upper bounds based on the requested CI level
    alpha = 1.0 - ci_level
    lower_bound = np.percentile(bootstrapped_proportions, (alpha / 2) * 100)
    upper_bound = np.percentile(bootstrapped_proportions, (1 - (alpha / 2)) * 100)

    # Format the output bounds
    ap_str = f"{mean_ap * 100:.3f}% (95% CI: [{lower_bound * 100:.3f}%, {upper_bound * 100:.3f}%])"

    # Compare to ETHZ's CAP using the mean
    ethz_area_proportion = compute_ethz_corrected_area_proportion(region)
    mean_delta = (mean_ap / ethz_area_proportion) - 1.0

    print(f"{region.title()}'s test subregion: Our cocoa area proportion (corrected): {ap_str} "
          f"({mean_delta * 100:.3f}% off ETHZ's estimate)")

    return mean_ap

# Execution
print_our_corrected_area_proportion('ashanti')
print_our_corrected_area_proportion('western north')
print_our_corrected_area_proportion('western')

Ashanti's test subregion: Our cocoa area proportion (corrected): 65.716% (95% CI: [64.806%, 66.614%]) (1.632% off ETHZ's estimate)
Western North's test subregion: Our cocoa area proportion (corrected): 98.095% (95% CI: [97.797%, 98.380%]) (-1.035% off ETHZ's estimate)
Western's test subregion: Our cocoa area proportion (corrected): 95.555% (95% CI: [95.218%, 95.861%]) (-2.017% off ETHZ's estimate)


np.float64(0.9555487470816632)